# Week 9: Fair Housing Compliance Demo

This notebook demonstrates the **Fair Housing Act** compliance deliverables:

1. **`ComplianceChecker`** with a pattern library for prohibited / risky terms (protected classes and proxies).
2. **Multi-level findings**: `error` (must fix before publish), `warning` (human review), `info` (awareness only).
3. **Quality targets** on the labelled test corpus: 100% recall on known violations, precision > 80% (validated in `tests/test_compliance_checker.py`).
4. **Team documentation**: `docs/fair_housing_guidelines.md`.
5. **Integration example**: `scripts/listing_submission_example.py` — we replay the submission workflow below.

Run this notebook from the repo root or from `notebooks/`; the first code cell adds the project to `sys.path`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.compliance_checker import ERROR, ComplianceChecker
from scripts.listing_submission_example import (
    Listing,
    ListingSubmissionError,
    ListingSubmissionService,
)
from tests.test_compliance_checker import CLEAN_LISTINGS, VIOLATING_LISTINGS

checker = ComplianceChecker()
print(f"Project root: {PROJECT_ROOT}")
print(f"Checker categories: {len(checker.categories)}")

## 1. Pattern library (categories)

Patterns are grouped by **Fair Housing** theme (race, religion, familial status, disability, source of income, steering, etc.). Each match returns a **severity**, a **message**, and a **suggestion** for rewriting copy.

In [ ]:
for cat in sorted(checker.categories):
    print(f"  • {cat}")

## 2. Three severity levels

| Level | Meaning | Typical workflow |
|-------|---------|------------------|
| **error** | Almost always non-compliant | Block publication until rewritten |
| **warning** | Context-dependent | Route to human reviewer |
| **info** | Legal but historically used for steering | Log; no block |

A listing is **`compliant`** when there are **no errors** (warnings and infos may still appear).

In [ ]:
def show_report(label: str, text: str) -> None:
    report = checker.check_listing(text)
    print("=" * 72)
    print(label)
    print(f"compliant={report.compliant}  errors={report.error_count}  "
          f"warnings={report.warning_count}  info={report.info_count}")
    print(checker.format_report(report))
    print()


# Error — blocks publication
show_report(
    "ERROR example (familial status + source of income)",
    "Adults only. No Section 8 vouchers.",
)

# Warning — review, does not set compliant=False by itself
show_report(
    "WARNING example (ideal-occupant language)",
    "Perfect for young professionals near downtown.",
)

# Info — awareness only
show_report(
    "INFO example (steering-adjacent phrasing)",
    "Located in a desirable neighborhood with great schools.",
)

# Clean listing
show_report(
    "Clean example",
    "3 bed, 2 bath ranch with updated kitchen and fenced backyard.",
)

## 3. Violation detail (`Violation` objects)

Each finding includes **category**, **severity**, **matched text**, **character offsets**, and a **suggestion**. Below: one violating string broken into a small table (optional: requires `pandas`).

In [ ]:
sample = "No children. Walking distance to the church."
rep = checker.check_listing(sample)

try:
    import pandas as pd

    rows = [v.__dict__ for v in rep.violations]
    display(pd.DataFrame(rows)[["category", "severity", "matched_text", "suggestion"]])
except ImportError:
    for v in rep.violations:
        print(v)

## 4. Labelled corpus: recall and precision

The test module defines **`VIOLATING_LISTINGS`** (ground-truth: must produce ≥1 **error**) and **`CLEAN_LISTINGS`** (ground-truth: **no** errors). We mirror the metric in the tests: **recall** = fraction of violating texts that triggered an error; **precision** = fraction of error-flagged texts that were truly violating.

In [ ]:
tp = fp = fn = tn = 0
for text in VIOLATING_LISTINGS:
    err = checker.check_listing(text).error_count > 0
    tp += err
    fn += not err
for text in CLEAN_LISTINGS:
    err = checker.check_listing(text).error_count > 0
    fp += err
    tn += not err

recall = tp / (tp + fn) if (tp + fn) else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0

print(f"Violating samples: {len(VIOLATING_LISTINGS)}  Clean samples: {len(CLEAN_LISTINGS)}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Recall (on violations):    {recall:.3f}  (target: 1.000)")
print(f"Precision (error vs all): {precision:.3f}  (target: > 0.80)")

assert recall == 1.0, "Recall must be 100% on known violations"
assert precision > 0.8, "Precision must exceed 80%"

## 5. Team documentation

Policy background, protected classes, severity semantics, and rewrite guidance live in:

**`docs/fair_housing_guidelines.md`**

Open that file in the editor or render it in your doc viewer; it is the canonical reference for *why* patterns exist and *how* editors should rewrite copy.

## 6. Integration: listing submission workflow

`ListingSubmissionService` (from `scripts/listing_submission_example.py`):

- **Errors** → `blocked`, raises `ListingSubmissionError`
- **Warnings only** → `pending_review` (review queue)
- **Clean or info-only** → `published`

Three sample listings below: one clean, one needs review, one blocked.

In [ ]:
service = ListingSubmissionService(checker)

demos = [
    Listing(
        "NB-001",
        "100 Main St",
        450_000,
        "2 bed condo with balcony and in-unit laundry near transit.",
    ),
    Listing(
        "NB-002",
        "200 Oak Ave",
        620_000,
        "Ideal for young professionals. Great schools nearby.",
    ),
    Listing(
        "NB-003",
        "300 Pine Rd",
        890_000,
        "Adults only. No Section 8.",
    ),
]

for listing in demos:
    print("=" * 72)
    print(listing.listing_id, listing.description[:60] + "..." if len(listing.description) > 60 else listing.description)
    try:
        service.submit(listing)
        print(f"  → status: {listing.status}")
    except ListingSubmissionError as e:
        print(f"  → blocked: {e}")
    for line in listing.audit_log[-3:]:
        print(f"     {line}")

print()
print("Published:", [x.listing_id for x in service.published])
print("Pending review:", [x.listing_id for x in service.review_queue])

## 7. JSON export (audit / API)

`ComplianceReport.to_dict()` is suitable for logging or API responses.

In [ ]:
report = checker.check_listing("English-speaking tenants only.")
print(json.dumps(report.to_dict(), indent=2)[:1200])
if len(json.dumps(report.to_dict())) > 1200:
    print("...")

## 8. Extending with `extra_patterns`

Add jurisdiction-specific rules without editing the core library:

In [ ]:
custom = ComplianceChecker(
    extra_patterns={
        "custom_rule": [
            (
                r"(?<![A-Za-z0-9])no\s+military\s+vouchers(?![A-Za-z0-9])",
                ERROR,
                "Hypothetical source-of-income exclusion",
                "Remove; verify with counsel.",
            ),
        ],
    }
)
r = custom.check_listing("We accept all applicants except no military vouchers.")
print(custom.format_report(r))